# CLAW summary tables and control ensemble changes

This notebook writes and displays the numerical tables used by the following figures:

1. State-level CLAW summaries: left-choice probability, decision time, and terminal probability.
2. Outgoing transition probabilities, including END transitions.
3. Changes in CBGT control ensemble and DDM parameters along observed state transitions.

Run this after Notebooks 01–03. 

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from spn_figures.config import DERIVED, SOURCE
from spn_figures.io import read_csv, write_csv
from spn_figures.claw import (
    binarize_activity,
    build_cbgt_state_activity,
    control_transition_scores,
    fit_control_ensembles,
    summarize_claw,
    transition_table_with_end,
    trial_sequences,
)

required_files = [
    DERIVED / "cbgt" / "trial_state_bins.csv.gz",
    DERIVED / "ibl" / "population_activity_bins.csv.gz",
    DERIVED / "steinmetz" / "population_activity_bins.csv.gz",
]

## 1. Build trial-level state walks

CBGT state bins are already binary. Empirical population activity is binarized session by session using the same thresholding logic as the original CLAW notebooks.

In [3]:
# CBGT: states are already binary in the data generated by Notebook 01.
cbgt_bins = read_csv(DERIVED / "cbgt" / "trial_state_bins.csv.gz")
cbgt_bins["dataset"] = "CBGT"
if "state" not in cbgt_bins.columns:
    cbgt_bins["state"] = sum(
        cbgt_bins[name].astype(int) * (2 ** (3 - index))
        for index, name in enumerate(["dSPN_left", "dSPN_right", "iSPN_left", "iSPN_right"])
    ).astype(int)

# Empirical datasets: use the exact population-activity tables from Notebooks 02 and 03.
ibl_activity = read_csv(DERIVED / "ibl" / "population_activity_bins.csv.gz")
ibl_activity["dataset"] = "IBL"
ibl_bins, ibl_thresholds = binarize_activity(ibl_activity)

steinmetz_activity = read_csv(DERIVED / "steinmetz" / "population_activity_bins.csv.gz")
steinmetz_activity["dataset"] = "Steinmetz"
steinmetz_bins, steinmetz_thresholds = binarize_activity(steinmetz_activity)

# Save the empirical binary time-bin tables and thresholds.
ibl_bins.to_csv(DERIVED / "ibl" / "state_bins.csv.gz", index=False)
steinmetz_bins.to_csv(DERIVED / "steinmetz" / "state_bins.csv.gz", index=False)
write_csv(ibl_thresholds, DERIVED / "ibl" / "binarization_thresholds.csv")
write_csv(steinmetz_thresholds, DERIVED / "steinmetz" / "binarization_thresholds.csv")

# Convert binned state tables to compressed trial walks.
cbgt_trials = trial_sequences(cbgt_bins)
ibl_trials = trial_sequences(ibl_bins)
steinmetz_trials = trial_sequences(steinmetz_bins)

cbgt_trials.to_csv(DERIVED / "cbgt" / "trial_sequences.csv.gz", index=False)
ibl_trials.to_csv(DERIVED / "ibl" / "trial_sequences.csv.gz", index=False)
steinmetz_trials.to_csv(DERIVED / "steinmetz" / "trial_sequences.csv.gz", index=False)

print("Trial sequences:")
print("CBGT:", len(cbgt_trials))
print("IBL:", len(ibl_trials))
print("Steinmetz:", len(steinmetz_trials))

Trial sequences:
CBGT: 15000
IBL: 752
Steinmetz: 386


## 2. Output state metrics and transition probabilities

`claw_nodes.csv` contains one row per state. `claw_transitions_with_end.csv` contains all outgoing transition probabilities, including transitions to `END`.

In [4]:
cbgt_nodes, cbgt_edges = summarize_claw(cbgt_trials, balance_choice=False)
ibl_nodes, ibl_edges = summarize_claw(ibl_trials, balance_choice=True)
steinmetz_nodes, steinmetz_edges = summarize_claw(steinmetz_trials, balance_choice=True)

summary_tables = {
    "cbgt": (cbgt_nodes, cbgt_edges, DERIVED / "cbgt"),
    "ibl": (ibl_nodes, ibl_edges, DERIVED / "ibl"),
    "steinmetz": (steinmetz_nodes, steinmetz_edges, DERIVED / "steinmetz"),
}

for name, (nodes, edges, folder) in summary_tables.items():
    transitions = transition_table_with_end(nodes, edges)
    write_csv(nodes, folder / "claw_nodes.csv")
    write_csv(edges, folder / "claw_edges.csv")
    write_csv(transitions, folder / "claw_transitions_with_end.csv")

    print(f"\n{name.upper()} state metrics")
    display(
        nodes[
            [
                "state",
                "bits",
                "n_trials",
                "left_choice_probability",
                "mean_decision_time_ms",
                "median_decision_time_ms",
                "terminal_probability",
                "end_count",
                "outgoing_total",
            ]
        ].sort_values("state")
    )

    print(f"{name.upper()} outgoing transitions, including END")
    display(transitions)


CBGT state metrics


,state,bits,n_trials,left_choice_probability,mean_decision_time_ms,median_decision_time_ms,terminal_probability,end_count,outgoing_total
0,0,"[0,0,0,0]",14999,0.504700,115.141009,100.0,0.076983,1191,15471
1,1,"[0,0,0,1]",1142,0.407180,170.551664,150.0,0.196594,254,1292
2,2,"[0,0,1,0]",1184,0.559966,169.290541,150.0,0.161265,209,1296
3,3,"[0,0,1,1]",1289,0.476338,181.737781,160.0,0.283007,448,1583
4,4,"[0,1,0,0]",4281,0.129175,110.665732,100.0,0.546377,2368,4334
5,5,"[0,1,0,1]",1464,0.119536,134.521858,110.0,0.682243,1022,1498
6,6,"[0,1,1,0]",234,0.243590,148.675214,130.0,0.412500,99,240
7,7,"[0,1,1,1]",1078,0.218924,175.519481,150.0,0.588380,719,1222
8,8,"[1,0,0,0]",4269,0.880300,111.091591,100.0,0.526085,2279,4332
9,9,"[1,0,0,1]",260,0.715385,146.692308,120.0,0.397727,105,264


CBGT outgoing transitions, including END


,source,source_bits,target,target_bits,transition_type,count,probability
0,0,"[0,0,0,0]",END,END,state_to_END,1191,0.076983
1,0,"[0,0,0,0]",1,"[0,0,0,1]",state_to_state,1061,0.068580
2,0,"[0,0,0,0]",2,"[0,0,1,0]",state_to_state,1105,0.071424
3,0,"[0,0,0,0]",3,"[0,0,1,1]",state_to_state,367,0.023722
4,0,"[0,0,0,0]",4,"[0,1,0,0]",state_to_state,4231,0.273479
...,...,...,...,...,...,...,...
202,15,"[1,1,1,1]",10,"[1,0,1,0]",state_to_state,5,0.003228
203,15,"[1,1,1,1]",11,"[1,0,1,1]",state_to_state,120,0.077469
204,15,"[1,1,1,1]",12,"[1,1,0,0]",state_to_state,2,0.001291
205,15,"[1,1,1,1]",13,"[1,1,0,1]",state_to_state,16,0.010329



IBL state metrics


...

,source,source_bits,target,target_bits,transition_type,count,probability
0,0,"[0,0,0,0]",END,END,state_to_END,382,0.278223
1,0,"[0,0,0,0]",1,"[0,0,0,1]",state_to_state,196,0.142753
2,0,"[0,0,0,0]",2,"[0,0,1,0]",state_to_state,265,0.193008
3,0,"[0,0,0,0]",3,"[0,0,1,1]",state_to_state,6,0.004370
4,0,"[0,0,0,0]",4,"[0,1,0,0]",state_to_state,198,0.144210
...,...,...,...,...,...,...,...
119,14,"[1,1,1,0]",6,"[0,1,1,0]",state_to_state,1,0.166667
120,14,"[1,1,1,0]",8,"[1,0,0,0]",state_to_state,1,0.166667
121,14,"[1,1,1,0]",10,"[1,0,1,0]",state_to_state,1,0.166667
122,14,"[1,1,1,0]",12,"[1,1,0,0]",state_to_state,2,0.333333



STEINMETZ state metrics


,state,bits,n_trials,left_choice_probability,mean_decision_time_ms,median_decision_time_ms,terminal_probability,end_count,outgoing_total
0,0,"[0,0,0,0]",386,0.500000,207.272451,189.452258,0.200820,147,732
1,1,"[0,0,0,1]",66,0.348485,226.001550,207.774918,0.222222,20,90
2,2,"[0,0,1,0]",167,0.592814,219.384049,204.415725,0.352941,78,221
3,3,"[0,0,1,1]",15,0.333333,237.161152,223.176097,0.187500,3,16
4,4,"[0,1,0,0]",127,0.267717,224.223802,213.300994,0.255952,43,168
5,5,"[0,1,0,1]",24,0.166667,233.712420,212.403697,0.406250,13,32
6,6,"[0,1,1,0]",57,0.298246,218.072681,212.450775,0.500000,33,66
7,7,"[0,1,1,1]",7,0.000000,270.037746,252.982793,0.125000,1,8
8,8,"[1,0,0,0]",129,0.496124,221.499065,210.101318,0.094527,19,201
9,9,"[1,0,0,1]",4,0.750000,218.098425,178.968616,0.000000,0,4


STEINMETZ outgoing transitions, including END


,source,source_bits,target,target_bits,transition_type,count,probability
0,0,"[0,0,0,0]",END,END,state_to_END,147,0.200820
1,0,"[0,0,0,0]",1,"[0,0,0,1]",state_to_state,73,0.099727
2,0,"[0,0,0,0]",2,"[0,0,1,0]",state_to_state,171,0.233607
3,0,"[0,0,0,0]",3,"[0,0,1,1]",state_to_state,4,0.005464
4,0,"[0,0,0,0]",4,"[0,1,0,0]",state_to_state,127,0.173497
...,...,...,...,...,...,...,...
94,13,"[1,1,0,1]",5,"[0,1,0,1]",state_to_state,1,0.333333
95,14,"[1,1,1,0]",END,END,state_to_END,3,0.428571
96,14,"[1,1,1,0]",4,"[0,1,0,0]",state_to_state,2,0.285714
97,14,"[1,1,1,0]",6,"[0,1,1,0]",state_to_state,1,0.142857


## 3. Changes in control ensembles and DDM parameters

This section reconstructs the 18-dimensional average CBGT activity vector for every observed network/state from the uploaded network folders, then projects each state-to-state transition through the original CCA loadings.

Required external matrices from https://doi.org/10.1371/journal.pcbi.1012966:

```text
data/source/cbgt/control/F_matrix.npy
data/source/cbgt/control/D_matrix.npy
```

In [5]:
def find_control_file(filename: str) -> Path:
    candidates = [
        SOURCE / "cbgt" / "control" / filename,
        SOURCE / "cbgt" / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Expected one of:" + chr(10)
        + chr(10).join(str(path) for path in candidates)
    )

f_path = find_control_file("F_matrix.npy")
d_path = find_control_file("D_matrix.npy")

f_matrix = np.load(f_path)
d_matrix = np.load(d_path)

print("Using:")
print("F matrix:", f_path, f_matrix.shape)
print("D matrix:", d_path, d_matrix.shape)

Using:
F matrix: /Users/zhuojunyu/Desktop/CBGTPy_control/All_GitHub_Code/data/source/cbgt/F_matrix.npy (300, 18)
D matrix: /Users/zhuojunyu/Desktop/CBGTPy_control/All_GitHub_Code/data/source/cbgt/D_matrix.npy (300, 4)


In [6]:
state_activity = build_cbgt_state_activity(
    SOURCE / "cbgt" / "networks",
    n_networks=300,
)
state_activity.to_csv(
    DERIVED / "cbgt" / "state_activity_18d.csv.gz",
    index=False,
)

print("State-activity rows:", len(state_activity))

State-activity rows: 3307


In [7]:
neural_loadings, ddm_loadings = fit_control_ensembles(f_matrix, d_matrix)
control_scores = control_transition_scores(
    state_activity,
    cbgt_edges,
    neural_loadings,
    ddm_loadings,
)

# Add bit labels for readability.
control_scores.insert(
    1,
    "source_bits",
    control_scores["source"].map(lambda state: f"[{int(state):04b}]"),
)
control_scores.insert(
    3,
    "target_bits",
    control_scores["target"].map(lambda state: f"[{int(state):04b}]"),
)

# Save one combined table and two narrower tables.
write_csv(control_scores, DERIVED / "cbgt" / "control_and_ddm_transition_changes.csv")

control_columns = [
    "source",
    "source_bits",
    "target",
    "target_bits",
    "probability",
    "n_common_networks",
    "choice",
    "responsiveness",
    "pliancy",
]
ddm_columns = [
    "source",
    "source_bits",
    "target",
    "target_bits",
    "probability",
    "n_common_networks",
    "a",
    "v",
    "t",
    "z",
]

control_values = control_scores[control_columns].copy()
ddm_values = control_scores[ddm_columns].copy()

write_csv(control_values, DERIVED / "cbgt" / "control_ensemble_transition_changes.csv")
write_csv(ddm_values, DERIVED / "cbgt" / "ddm_parameter_transition_changes.csv")

print("Control ensemble changes along observed CBGT state transitions")
display(control_values.sort_values(["source", "target"]))

print("DDM parameter changes along observed CBGT state transitions")
display(ddm_values.sort_values(["source", "target"]))

Control ensemble changes along observed CBGT state transitions


,source,source_bits,target,target_bits,probability,n_common_networks,choice,responsiveness,pliancy
5,0,[0000],1,[0001],0.068580,176,-19.431045,29.042237,44.288665
6,0,[0000],2,[0010],0.071424,177,8.188142,36.013898,44.008519
14,0,[0000],3,[0011],0.023722,109,-5.217845,41.688376,52.373002
1,0,[0000],4,[0100],0.273479,292,-24.339603,31.412218,40.807639
2,0,[0000],5,[0101],0.036003,266,-34.390579,43.082977,51.046748
...,...,...,...,...,...,...,...,...,...
186,15,[1111],10,[1010],0.003228,193,29.149798,-15.770779,-11.615022
180,15,[1111],11,[1011],0.077469,127,14.233804,-10.045788,-4.128788
185,15,[1111],12,[1100],0.001291,160,0.419351,-15.593644,-11.467699
187,15,[1111],13,[1101],0.010329,152,-11.936469,-9.822391,-5.165102


DDM parameter changes along observed CBGT state transitions


,source,source_bits,target,target_bits,probability,n_common_networks,a,v,t,z
5,0,[0000],1,[0001],0.068580,176,-2.975214,-24.411418,-60.946812,10.480973
6,0,[0000],2,[0010],0.071424,177,-4.813066,6.996765,-59.626370,12.232368
14,0,[0000],3,[0011],0.023722,109,-6.637692,-8.501950,-72.584981,13.569709
1,0,[0000],4,[0100],0.273479,292,-6.521266,-28.981275,-60.563730,9.540649
2,0,[0000],5,[0101],0.036003,266,-11.848643,-40.695568,-79.206901,11.801734
...,...,...,...,...,...,...,...,...,...,...
186,15,[1111],10,[1010],0.003228,193,9.396956,32.290721,26.738892,-1.797145
180,15,[1111],11,[1011],0.077469,127,7.055224,15.267907,12.467538,-0.714172
185,15,[1111],12,[1100],0.001291,160,6.252638,0.328175,20.610314,-3.223974
187,15,[1111],13,[1101],0.010329,152,3.515959,-13.677430,8.888049,-2.202863
